In [2]:
import keras
import tensorflow as tf
from os import listdir
import os
import numpy as np
from PIL import Image 

2025-06-30 13:45:11.682261: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-30 13:45:11.690978: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751305511.701124  541388 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751305511.704090  541388 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751305511.711502  541388 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
def load_image_and_mask(image_path, mask_path, image_size=(1024, 1024)):
    # Load and preprocess image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32) / 255.0

    # Load and preprocess mask 
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (256, 256), method='nearest')  
    mask = tf.cast(mask, tf.uint8)
    filename = tf.strings.split(image_path, os.sep)[-1]
    return image, mask, filename


image_dir = "/home/research/Documents/eco-seg/example_dataset/images/"
mask_dir = "/home/research/Documents/eco-seg/example_dataset/labels/"


# Get sorted lists of image and mask paths (filenames must match)
image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir) ])
mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])


dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
dataset = dataset.map(lambda img, msk: load_image_and_mask(img, msk), num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(8).prefetch(tf.data.AUTOTUNE)

# Example: iterate through the dataset
for images, masks,filename in dataset.take(2):
    print("Images batch shape:", images.shape)
    print("Masks batch shape:", masks.shape)
    print("Filenames:", filename.numpy())
    print("Unique mask values:", tf.unique(tf.reshape(masks, [-1]))[0].numpy())

I0000 00:00:1751305513.209975  541388 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18247 MB memory:  -> device: 0, name: NVIDIA RTX 4000 Ada Generation, pci bus id: 0000:01:00.0, compute capability: 8.9


Images batch shape: (8, 1024, 1024, 3)
Masks batch shape: (8, 256, 256, 1)
Filenames: [b'S-1_D062420_081920_60_I282_NE_L6.JPG'
 b'S14400_D061020_071520_48_I512_NE_L6.jpg'
 b'S14400_D071520_090220_48_I986_NE_L6.JPG'
 b'S14687_D072220_091620_20_I1041_NE_L6.JPG'
 b'S14687_D091620_110920_20_I1055_NE_L6.JPG'
 b'S15091_D062420_081920_42_I267_NE_L6.JPG'
 b'S15192_D081220_100720_52_I365_NE_L6.JPG'
 b'S15192_D100720_111820_52_I558_NE_L6.JPG']
Unique mask values: [  0 149  76]
Images batch shape: (8, 1024, 1024, 3)
Masks batch shape: (8, 256, 256, 1)
Filenames: [b'S15193_D100720_111820_17_I151_NE_L6.JPG'
 b'S15499_D032420_061820_8_I482_NE_L6.JPG'
 b'S15796_D110420_061621_58_I3567_NE_L6.JPG'
 b'S16139_D110920_042721_36_I930_NE_L6.JPG'
 b'S16880_D100720_111820_47_I554_NE_L6.JPG'
 b'S18421_D062420_081920_37_I497_NE_L6.JPG'
 b'S18720_D110420_061621_63_I496_NE_L6.JPG'
 b'S19091_D100720_040821_16_I1904_NE_L6.JPG']
Unique mask values: [ 0 76]


2025-06-30 13:45:13.639858: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [4]:
def add_sam_dummy_inputs(image, mask, filename):
    # image: (1024, 1024, 3)
    # mask: (1024, 1024, 1)
    # filename: string
    batch_size = tf.shape(image)[0]
    return {
        'images': image,
        'points': tf.zeros((batch_size, 0, 2), dtype=tf.float32),
        'labels': tf.zeros((batch_size, 0), dtype=tf.float32),
        'boxes': tf.zeros((batch_size, 0, 2, 2), dtype=tf.float32),
        'masks': tf.zeros((batch_size, 0, 0, 0, 1), dtype=tf.float32),
    }, mask

dataset = dataset.map(add_sam_dummy_inputs, num_parallel_calls=tf.data.AUTOTUNE)

In [2]:
import keras_hub
base_model = keras_hub.models.SAMImageSegmenter.from_preset('sam_base_sa1b')
base_model.trainable = False



In [6]:
inputs = base_model.input
outputs = base_model(inputs) # use the output of the base model as the input of the new model


In [7]:
# Add one additional Conv2D layer and an output layer for water segmentation
if isinstance(outputs, dict):
    masks = outputs["masks"]
elif isinstance(outputs, (list, tuple)):
    _, masks = outputs
else:
    raise ValueError("Unexpected output type from base_model")

# Select the first mask (assuming it's the relevant one for water)
mask = masks[:, 0, :, :]  
#print(mask.shape)
mask = keras.layers.Reshape((256, 256, 1))(mask)

# Additional trainable layer
x = keras.layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu')(mask)
output = keras.layers.Conv2D(1, kernel_size=(1, 1), activation='sigmoid')(x)

model = keras.Model(inputs=inputs, outputs=output)
model.summary()

Model: "functional_22"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ boxes (InputLayer)  │ (None, None, 2,   │          0 │ -                 │
│                     │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ images (InputLayer) │ (None, 1024,      │          0 │ -                 │
│                     │ 1024, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ labels (InputLayer) │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masks (InputLayer)  │ (None, None,      │          0 │ -                 │
│                     │ None, None, 1)    │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ points (InputLayer) │ (None, None, 2)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sam_image_segmenter │ [(None, 4),       │ 93,735,728 │ boxes[0][0],      │
│ (SAMImageSegmenter) │ (None, 4, 256,    │            │ images[0][0],     │
│                     │ 256)]             │            │ labels[0][0],     │
│                     │                   │            │ masks[0][0],      │
│                     │                   │            │ points[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 256, 256)  │          0 │ sam_image_segmen… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 256, 256,  │          0 │ get_item[0][0]    │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 256, 256,  │        640 │ reshape[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 256, 256,  │         65 │ conv2d_6[0][0]    │
│                     │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 93,736,433 (357.58 MB)

 Trainable params: 705 (2.75 KB)

 Non-trainable params: 93,735,728 (357.57 MB)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                loss=keras.losses.BinaryCrossentropy(),
                metrics=[keras.metrics.BinaryAccuracy()])
# Train the model
model.fit(dataset, epochs=100, verbose=1)
# Save the model
model.save("water_segmentation_model.keras")


Epoch 1/100


I0000 00:00:1751305524.235862  541473 service.cc:152] XLA service 0x7fb338012340 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1751305524.235981  541473 service.cc:160]   StreamExecutor device (0): NVIDIA RTX 4000 Ada Generation, Compute Capability 8.9
2025-06-30 13:45:24.713168: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1751305526.850039  541473 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-06-30 13:45:28.058106: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_142', 4 bytes spill stores, 4 bytes spill loads

2025-06-30 13:45:28.661406: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1

2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 879ms/step - binary_accuracy: 0.3688 - loss: 1.9462

2025-06-30 13:46:07.197147: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1582', 388 bytes spill stores, 388 bytes spill loads

2025-06-30 13:46:07.283581: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1582', 388 bytes spill stores, 388 bytes spill loads

2025-06-30 13:46:09.075600: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5', 5068 bytes spill stores, 5080 bytes spill loads

2025-06-30 13:46:09.644944: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5', 716 bytes spill stores, 592 bytes spill loads

2025-06-30 13:46:09.766467: I extern

3/3 ━━━━━━━━━━━━━━━━━━━━ 85s 19s/step - binary_accuracy: 0.3714 - loss: 1.9930
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 621ms/step - binary_accuracy: 0.3413 - loss: 1.7150
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 623ms/step - binary_accuracy: 0.2886 - loss: 1.4481
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 625ms/step - binary_accuracy: 0.2135 - loss: 1.1829
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 621ms/step - binary_accuracy: 0.1482 - loss: 0.9183
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 622ms/step - binary_accuracy: 0.1017 - loss: 0.6542
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 622ms/step - binary_accuracy: 0.0684 - loss: 0.3903
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 619ms/step - binary_accuracy: 0.0462 - loss: 0.1268
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 620ms/step - binary_accuracy: 0.0322 - loss: -0.1364
Epoch 10/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 621ms/step - binary_accuracy: 0.0224 - loss: -0.3993
Epoch 11/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 623ms/step - binary_accuracy: 0.0156 - loss: -0.6